# Comparison of ATS and non-ATS results
Reduces the other notebooks to the base function needed to compare behaviour with and without ATS for shaped streams (prio 5). 

## Imports and Function definitions

In [112]:
import pandas as pd
import numpy as np
import  matplotlib.pyplot as plt
import re
from omnetpp.scave import results, chart, utils
%matplotlib inline  

### Functions max latencies

In [113]:
def get_device(module):
    return module.split(".")[1]

def get_streamId(module, df):
    # assumning that streamIds are deterministic
    attrname = "*." + module.split(".")[1] + "." + module.split(".")[2] + ".display-name"
    attrval = df.loc[df['attrname'] == attrname,'attrvalue']
    try:
        return attrval.to_list()[0].split(" ")[0]
    except:
        return " "

def streamencodings(df):
    df_notna = df[df['attrname'].notna()]
    all_encodings = df_notna.loc[df_notna['attrname'].str.contains(".bridging.streamCoder.encoder.mapping")]
    # build a single json file based on streamname as key
    encodings = dict()
    for idx, row in all_encodings.iterrows():
        dev = row['attrname']
        encoding_str = row['attrvalue']
        encoding = re.findall(r'\{([^}]*)\}', encoding_str)
        for item in encoding:
            enc = item.split(",")  
            for e in enc:
                if "stream" in e:
                    stream = re.findall('"([^"]*)"', e)[0]
                elif "pcp" in e:
                    pcp = re.findall(r'\d+', e)[0]
                elif "vlan" in e:
                    vlan = re.findall(r'\d+', e)[0]
            # add infos to stream encoding
            if stream in encodings.keys():
                if pcp not in encodings[stream]['pcp']:
                    encodings[stream]['pcp'].append(pcp)
                if vlan not in encodings[stream]['vlan']:
                    encodings[stream]['vlan'].append(vlan)
                if dev not in encodings[stream]['dev']:
                    encodings[stream]['dev'].append(dev)
            else:
                encodings[stream] = {'pcp' : [pcp],
                                    'vlan' : [vlan],
                                    'dev' : [dev]}
    return encodings
        
def set_streamId(name): # streamnames are only sometimes deterministic, these are the special cases
    if "_" in name: # CAN
        return "SControl"
    if name == "SReset": # a single stream that is not formatted the same
        return "SControl"
    if name == " ": # Tcp-Stream
        return "SEtsiCamOut"
    
    return name

def get_pcp(streamname, device , df):
    if "zonalController" in device:
        attrname = "*.zonalController*.bridging.streamCoder.decoder.mapping"
    else:
        attrname = "*." + device + ".bridging.streamCoder.decoder.mapping"
    
    try:
        decoding_str = df.loc[df['attrname']==attrname, 'attrvalue'].to_list()[0]
        decoding = re.findall(r'\{([^}]*)\}', decoding_str)
        # get pcp for the stream from the table
        for item in decoding:
            if (streamname + "\"") in item:
                return item.split(",")[1][-1:]
    except:
        return " "

def get_pcp_from_encodings(streamname, encodings):
    try:
        return encodings[streamname]['pcp'][0]
    except:
        return " "
        

def max_delay(vals):
    try:
        return vals.max()
    except Exception as e:
        print(e)
        return "ERROR"


def extract_e2edelay(df):
    res = df[['runID','module', 'vectime', 'vecvalue']].dropna()
    res['device'] = res.apply(lambda row: get_device(row['module']), axis=1)
    res['streamname'] = res.apply(lambda row: get_streamId(row['module'], df), axis=1)
    res['streamname-Control'] = res.apply(lambda row: set_streamId(row['streamname']), axis=1)
    encodings  = streamencodings(df)
    res['pcp'] = res.apply(lambda row: get_pcp_from_encodings(row['streamname-Control'], encodings), axis=1)
    res['max e2e delay'] = res.apply(lambda row: max_delay(row['vecvalue']), axis=1)
    
    return res

### Functions queueing

In [114]:
def get_devicePort(module):
    split = module.split(".")
    return split[1] + "." + split[2]

def get_queueNo(module):
    split = module.split(".")
    if len(split) < 6:
        return "all"
    else:
        return split[5]


def extract_maxq(df):
    res = df[['runID', 'module', 'value']].dropna()
    res.sort_values(by='value', ascending=False, inplace=True)
#res_maxq['device'] = res_maxq.apply(lambda row: get_device(row['module']), axis=1)
    res['device+port'] = res.apply(lambda row: get_devicePort(row['module']), axis=1)
    res['queue'] = res.apply(lambda row: get_queueNo(row['module']), axis=1)
    res = res[['runID','device+port', 'queue', 'value']]
    return res

### Functions packet counts

In [115]:
def sourceorsink(attrname):
    if "localPort" in attrname:
        return "sink"
    elif "destPort" in attrname:
        return "source"
    elif "connectPort" in attrname: # this is TCP client
        return "source" 

def appports(df):
    r = df[['attrname', 'attrvalue']].dropna()
    info = r.loc[r['attrname'].str.contains("localPort|destPort|connectPort")].copy()
    #d = df.loc[df['attrname'].str.contains("destPort"), ['attrname', 'attrvalue']].copy()
    #c = df.loc[df['attrname'].str.contains("connectPort"), ['attrname', 'attrvalue']].copy()
    #info = pd.concat([l,d])
    info['deviceapp'] = info.apply(lambda row: get_devicePort(row['attrname']), axis=1)
    info['sourcesink'] = info.apply(lambda row: sourceorsink(row['attrname']), axis=1)
    info['port'] = info['attrvalue']
    
    return dict(zip(info.deviceapp, info.port))

def extract_numpacks(df):
    res = df[['runID', 'module', 'value']].dropna()
    res['device'] = res.apply(lambda row: get_device(row['module']), axis=1)
    res['device+app'] = res.apply(lambda row: get_devicePort(row['module']), axis=1)
    res['streamname'] = res.apply(lambda row: get_streamId(row['module'], df), axis=1)
    # get the application port (local and/or destination) to match apps later
    app_ports = appports(df)
    res['app-port'] = res.apply(lambda row: app_ports[row['device+app']], axis=1)
    return res


### Functions dropped packets

In [168]:
def get_dropreason(module):
    submod =  module.split(".")[-1]
    if "flowFilter" in submod:
        return "eligibility time"
    elif "streamFilter" in submod:
        return "SDU"

def get_filterno(module):
    return module.split(".")[-1]

def extract_droppedpackets(df):
    res = df[['runID','module', 'vectime', 'vecvalue']].dropna()
    res['device'] = res.apply(lambda row: get_device(row['module']), axis=1)
    res['filter-no'] = res.apply(lambda row: get_filterno(row['module']), axis=1) 
    res['dropreason'] = res.apply(lambda row: get_dropreason(row['module']), axis=1)
    res['streamname'] = res.apply(lambda row: get_streamId(row['module'], df), axis=1)
    res['sum-droppedpackets'] = res.apply(lambda row: sum(row['vecvalue']), axis=1)

    return res

## Config names

In [147]:
config_ats_vec = "TestMRTWithAnomaly-*.vec"#"Baseline_ATS-*.vec"
config_ats_sca = "TestMRTWithAnomaly-*.sca" #"Baseline_ATS-*.sca"
config_vec = "Testscenario-*.vec"#"Baseline-*.vec"
config_sca = "Testscenario-*.sca"#"Baseline-*.sca"


## Latencies and jitter

In [148]:
# read results

res = results.read_result_files(filenames = config_vec,
                                filter_expression = "name =~ meanBitLifeTimePerPacket:vector")
res_ats = results.read_result_files(filenames = config_ats_vec,
                                filter_expression = "name =~ meanBitLifeTimePerPacket:vector")
latencies = extract_e2edelay(res)
latencies_ats = extract_e2edelay(res_ats)

### Witout ATS

In [ ]:
latency_jitter_by_priority = []
for prio, group in latencies.groupby('pcp'):
    sorted = group.sort_values('max e2e delay', ascending = False)
    
    max_latency = group['max e2e delay'].max()
    min_latency = group['max e2e delay'].min()
    jitter = max_latency - min_latency

    max_stream = group.loc[group['max e2e delay'] == max_latency, 'module'].to_list()

    latency_jitter_by_priority.append({"pcp": prio,
                                       "max latency [s]": max_latency,
                                       "min latency [s]": min_latency,
                                       "jitter [s]": jitter,
                                       "stream with max latency": max_stream
                                      })

df_latency_prio = pd.DataFrame(latency_jitter_by_priority)

df_latency_prio

In [ ]:
l_p5  = latencies.loc[latencies['pcp'] == "5"].copy()

l_p5['min e2e delay'] = l_p5.apply(lambda row: row['vecvalue'].min(), axis=1)
l_p5['jitter'] = l_p5.apply(lambda row: row['max e2e delay'] - row['min e2e delay'], axis=1)

l_p5_streams = l_p5[['module','device', 'streamname', 'max e2e delay', 'min e2e delay', 'jitter']].sort_values('max e2e delay', ascending=False)
l_p5_streams.rename(columns= {'max e2e delay': 'max e2e delay [s]', 'min e2e delay': 'min e2e delay [s]', 'jitter': 'jitter [s]'}, inplace=True)

l_p5_streams

### With ATS

In [ ]:
latency_ats_jitter_by_priority = []
for prio, group in latencies_ats.groupby('pcp'):
    sorted = group.sort_values('max e2e delay', ascending = False)
    
    max_latency = group['max e2e delay'].max()
    min_latency = group['max e2e delay'].min()
    jitter = max_latency - min_latency

    max_stream = group.loc[group['max e2e delay'] == max_latency, 'module'].to_list()

    latency_ats_jitter_by_priority.append({"pcp": prio,
                                       "max latency [s]": max_latency,
                                       "min latency [s]": min_latency,
                                       "jitter [s]": jitter,
                                       "stream with max latency": max_stream
                                      })

df_latency_ats_prio = pd.DataFrame(latency_ats_jitter_by_priority)

df_latency_ats_prio

In [ ]:
l_p5_ats  = latencies_ats.loc[latencies_ats['pcp'] == "5"].copy()

l_p5_ats['min e2e delay'] = l_p5_ats.apply(lambda row: row['vecvalue'].min(), axis=1)
l_p5_ats['jitter'] = l_p5_ats.apply(lambda row: row['max e2e delay'] - row['min e2e delay'], axis=1)

l_p5_ats_streams = l_p5_ats[['module','device', 'streamname', 'max e2e delay', 'min e2e delay', 'jitter']].sort_values('max e2e delay', ascending=False)
l_p5_ats_streams.rename(columns= {'max e2e delay': 'max e2e delay [s]', 'min e2e delay': 'min e2e delay [s]', 'jitter': 'jitter [s]'}, inplace=True)

l_p5_ats_streams

## Queue lengths

In [153]:
# read results
res_q = results.read_result_files(filenames = config_sca,
                                filter_expression = "module =~ *.eth[*].macLayer.queue* AND name =~ queueLength:max")
res_q_ats = results.read_result_files(filenames = config_ats_sca,
                                filter_expression = "module =~ *.eth[*].macLayer.queue* AND name =~ queueLength:max")
max_q = extract_maxq(res_q)
max_q_ats = extract_maxq(res_q_ats)

### Without ATS

In [ ]:
qs = []
for q, group in max_q.groupby('queue'):
    max = group['value'].max()
    port = group.loc[group['value'] == max]
    ports = port['device+port'].to_list()
    if q == "all":
       qs.append({"prio": "all",
                "max": max,
                 "port": ports}) 
    else:
        prio = re.findall(r'\d+', q)[0]
        qs.append({"prio": prio,
                   "max": max,
                   "port": ports})

df_qs = pd.DataFrame(qs)

df_qs

### With ATS

In [ ]:
qs_ats = []
for q, group in max_q_ats.groupby('queue'):
    max = group['value'].max()
    port = group.loc[group['value'] == max]
    ports = port['device+port'].to_list()
    if q == "all":
       qs_ats.append({"prio": "all",
                "max": max,
                 "port": ports}) 
    else:
        prio = re.findall(r'\d+', q)[0]
        qs_ats.append({"prio": prio,
                   "max": max,
                   "port": ports})

df_qs_ats = pd.DataFrame(qs_ats)

df_qs_ats

## Missing frames
Checking if the same number of frames that is sent arrives at receiving app.

In [156]:
res_sources = results.read_result_files(filenames = config_sca,
                                 filter_expression = "module=~ *.io AND name =~ packetSent:count")
res_sinks = results.read_result_files(filenames = config_sca,
                                 filter_expression = "module =~ *.io AND name =~ packetReceived:count")
res_sources_ats = results.read_result_files(filenames = config_ats_sca,
                                 filter_expression = "module=~ *.io AND name =~ packetSent:count")
res_sinks_ats = results.read_result_files(filenames = config_ats_sca,
                                 filter_expression = "module =~ *.io AND name =~ packetReceived:count")

In [157]:
d_sources = extract_numpacks(res_sources)
# cleanup for OMNeT-v6.0.2 , because the statistics are initialized without data (so all apps are in res_sources)
# remove apps with value 0 from sources
d_sources = d_sources[d_sources['value']!= 0.0]
d_sources['srcsink'] = "source"
d_sinks = extract_numpacks(res_sinks)
# cleanup for OMNet-v6.0.2: "true" sinks have a value in column 'streamname'
d_sinks = d_sinks[d_sinks['streamname']!= " "]
d_sinks['srcsink'] = "sink"

numpack = pd.concat([d_sources, d_sinks])

In [158]:
d_sources_ats = extract_numpacks(res_sources_ats)
# cleanup for OMNeT-v6.0.2 , because the statistics are initialized without data (so all apps are in res_sources)
# remove apps with value 0 from sources
d_sources_ats = d_sources_ats[d_sources_ats['value']!= 0.0]
d_sources_ats['srcsink'] = "source"
d_sinks_ats = extract_numpacks(res_sinks_ats)
# cleanup for OMNet-v6.0.2: "true" sinks have a value in column 'streamname'
d_sinks_ats = d_sinks_ats[d_sinks_ats['streamname']!= " "]
d_sinks_ats['srcsink'] = "sink"

numpack_ats = pd.concat([d_sources_ats, d_sinks_ats])

### Without ATS

In [ ]:
# sort everything into stream - num at source - [num at sinks]
by_sink = []
for appport, group in d_sinks.groupby('app-port'):
    src = d_sources.loc[d_sources['app-port'] == appport]
    srcapp = src['device+app'].to_list()[0]
    #srcdvc = src['device'].to_list()[0]
    srcpcks = src['value'].to_list()[0]
    for idx, row in group.iterrows():
        by_sink.append({ "app-port": appport,
                        "stream name": row['streamname'],
                        "source app": srcapp,
                        #"source device": srcdvc,
                        "destination app": row['device+app'],
                        #"destination device": row['device'],
                        "packets sent [count]": srcpcks,
                        "packets received [count]": row['value'],
                        "packets difference": srcpcks - row['value']})
packet_diff = pd.DataFrame(by_sink)

packet_diff.sort_values('packets difference', ascending=False)

### With ATS

In [ ]:
# sort everything into stream - num at source - [num at sinks]
by_sink_ats = []
for appport, group in d_sinks_ats.groupby('app-port'):
    src = d_sources_ats.loc[d_sources_ats['app-port'] == appport]
    srcapp = src['device+app'].to_list()[0]
    #srcdvc = src['device'].to_list()[0]
    srcpcks = src['value'].to_list()[0]
    for idx, row in group.iterrows():
        by_sink_ats.append({ "app-port": appport,
                        "stream name": row['streamname'],
                        "source app": srcapp,
                        #"source device": srcdvc,
                        "destination app": row['device+app'],
                        #"destination device": row['device'],
                        "packets sent [count]": srcpcks,
                        "packets received [count]": row['value'],
                        "packets difference": srcpcks - row['value']})
packet_diff_ats = pd.DataFrame(by_sink_ats)

packet_diff_ats.sort_values('packets difference', ascending=False)

## Dropped packets

### With ATS

In [ ]:
res_drop_filter_ats = results.read_result_files(filenames = config_ats_vec,
                                 filter_expression = "module=~ *.streamFilter.* AND name =~ droppedPacketLengths:vector")

if not res_drop_filter_ats.empty:
    drop_filter_ats =  extract_droppedpackets(res_drop_filter_ats)
    display(drop_filter_ats[['module','filter-no','dropreason','sum-droppedpackets']])
else:
    print("no dropped frames")